In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]   # adjust if needed
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.path.append("../../src")

In [ ]:
PDF_PATH = pdf_path = "../../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
MODEL_NAME = model_name = "openai/gpt-oss-20b"  # "gpt-oss:20b"

SEED_ONTOLOGY_INPUT = str("../../data/ontology/ContextOntology-COInd4.owl")
ONTOLOGY_PATH = "../../data/ontology/ContextOntology-COInd4.owl"

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
import sys

ENV_PATH = Path.cwd().resolve().parents[1] / ".env"
load_dotenv(ENV_PATH)

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY", "")

if not OPENROUTER_KEY:
    raise ValueError(f"OPENROUTER_API_KEY not found in {ENV_PATH}")

sys.path.append("examples/approaches/SinglePassLLM")

# NeoOLAF on RAGTree datasets

This notebook runs the NeoOLAF RAGTree benchmark script using the same NeoOLAF code/library, but with dataset-specific guidance profiles.

DocRED is active by default. The other four datasets are present below as commented blocks. Start with a small `MAX_DOCS` value, then increase once the run is stable.

In [ ]:
# Shared runtime configuration
BACKEND = "openrouter"
HOST = OPENROUTER_HOST
API_KEY = OPENROUTER_KEY
MODEL_NAME = "openai/gpt-oss-20b"

# Parallelism
DOCUMENT_WORKERS = 2      # number of documents processed in parallel
MAX_WORKERS = 1           # intra-document/chunk workers, kept low for predictable calls

# OpenRouter/gpt-oss reasoning-output controls.
# gpt-oss providers can spend output tokens on reasoning and return empty final content
# if the output budget is too small. Keep reasoning minimal and leave room for JSON.
MAX_TOKENS = 8192
OPENROUTER_REASONING_EFFORT = "minimal"

# Quick test. Set to None for the full split.
MAX_DOCS = 5
MAX_DOCS_ARG = f"--max-docs {MAX_DOCS}" if MAX_DOCS is not None else ""

# Progress/error diagnostics
NO_TQDM = False
SHOW_ERROR_TRACEBACK = False
NO_TQDM_ARG = "--no-tqdm" if NO_TQDM else ""
SHOW_ERROR_TRACEBACK_ARG = "--show-error-traceback" if SHOW_ERROR_TRACEBACK else ""

# No-chunk setup: one very large chunk per normalized document.
CHUNK_SIZE = 10_000_000
CHUNK_OVERLAP = 0
MAX_CHUNKS = 1

# Conservative output limits. Increase later if recall is too low.
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = 20

## DocRED, active

In [ ]:
DATASET_JSONL = "../../../ragtree/data/preprocessed/docred_causal.jsonl"
ONTOLOGY_PATH = "../../../ragtree/data/ontology/DocREDOntology/ontology.ttl"
OUTPUT_JSONL = "./runs/neoolaf_docred_predictions.docred_constrained.canonical.jsonl"
SUMMARY_OUTPUT = "./runs/neoolaf_docred_eval_summary.json"
RUN_SUMMARY_OUTPUT = "./runs/neoolaf_docred_run_summary.json"
ERROR_LOG_JSONL = "./runs/neoolaf_docred_errors.jsonl"
SMOKE_GOLD_JSONL = "./runs/neoolaf_docred_smoke_gold.jsonl"
USER_GUIDANCE_PATH = "./configs/guidance_docred.json"


In [ ]:
# Build a matching gold file for smoke tests.
# If MAX_DOCS is not None, evaluating against the full dev set will show many missing_predictions.
# This subset keeps evaluation aligned with the documents actually processed.
import json
from pathlib import Path

EVAL_GOLD_JSONL = DATASET_JSONL
if MAX_DOCS is not None:
    selected = []
    with open(DATASET_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            split = str(row.get("type") or row.get("split") or "").lower()
            if split == "dev":
                selected.append(row)
            if len(selected) >= MAX_DOCS:
                break
    Path(SMOKE_GOLD_JSONL).parent.mkdir(parents=True, exist_ok=True)
    with open(SMOKE_GOLD_JSONL, "w", encoding="utf-8") as f:
        for row in selected:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    EVAL_GOLD_JSONL = SMOKE_GOLD_JSONL
    print(f"Smoke-test gold subset written: {EVAL_GOLD_JSONL} docs={len(selected)}")
else:
    print(f"Full evaluation gold: {EVAL_GOLD_JSONL}")


In [ ]:
!python ../../experiments/methods/run_neoolaf.py \
  --dataset-jsonl-path "{DATASET_JSONL}" \
  --ontology-path "{ONTOLOGY_PATH}" \
  --output-jsonl-path "{OUTPUT_JSONL}" \
  --error-log-jsonl-path "{ERROR_LOG_JSONL}" \
  --summary-output-path "{RUN_SUMMARY_OUTPUT}" \
  --backend-name "{BACKEND}" \
  --host "{HOST}" \
  --api-key "{API_KEY}" \
  --model-name "{MODEL_NAME}" \
  --max-tokens {MAX_TOKENS} \
  --openrouter-reasoning-effort {OPENROUTER_REASONING_EFFORT} \
  --openrouter-exclude-reasoning \
  --type-filter dev \
  --user-guidance-path "{USER_GUIDANCE_PATH}" \
  --few-shot-from-dataset \
  --few-shot-source-type dev \
  --few-shot-k 1 \
  --output-format canonical \
  --artifacts-root "./runs/neoolaf_docred_artifacts_docred_constrained" \
  --chunk-size {CHUNK_SIZE} \
  --chunk-overlap {CHUNK_OVERLAP} \
  --max-chunks {MAX_CHUNKS} \
  --max-expressions {MAX_EXPRESSIONS} \
  --max-relation-mentions {MAX_RELATION_MENTIONS} \
  --max-workers {MAX_WORKERS} \
  --document-workers {DOCUMENT_WORKERS} \
  --no-web-search \
  --no-checkpoints \
  --no-chunk-checkpoints \
  --no-resume \
  {MAX_DOCS_ARG}

In [ ]:
# Inspect NeoOLAF run diagnostics before the gold evaluator.
import json
from pathlib import Path

for path in [RUN_SUMMARY_OUTPUT, ERROR_LOG_JSONL]:
    p = Path(path)
    print("=", p)
    if not p.exists():
        print("not found")
        continue
    if p.suffix == ".json":
        print(json.dumps(json.loads(p.read_text(encoding="utf-8")), indent=2, ensure_ascii=False)[:8000])
    else:
        lines = p.read_text(encoding="utf-8").splitlines()
        print(f"lines={len(lines)}")
        for line in lines[:5]:
            obj = json.loads(line)
            print(json.dumps({
                "document_id": obj.get("document_id"),
                "error_type": obj.get("error_type"),
                "error_message": obj.get("error_message"),
                "artifact_dir": obj.get("artifact_dir"),
                "artifact_error_files": obj.get("artifact_error_files", [])[:2],
            }, indent=2, ensure_ascii=False)[:3000])


In [ ]:
!python ../../experiments/evaluation/eval_relations.py \
  --gold "{EVAL_GOLD_JSONL}" \
  --pred "{OUTPUT_JSONL}" \
  --output "{SUMMARY_OUTPUT}"

In [ ]:
import json
from pathlib import Path

summary_path = Path(SUMMARY_OUTPUT)
if summary_path.exists():
    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)
    print(json.dumps(summary, indent=2, ensure_ascii=False)[:5000])
else:
    print(f"Summary not found yet: {summary_path}")

## CausalBank, commented

In [ ]:
# DATASET_JSONL = "../../../ragtree/data/preprocessed/causalbank.jsonl"
# ONTOLOGY_PATH = "../../../ragtree/data/ontology/Wordnet-Synset/wordnet-synset.ttl"
# OUTPUT_JSONL = "./runs/neoolaf_causalbank_predictions.canonical.jsonl"
# SUMMARY_OUTPUT = "./runs/neoolaf_causalbank_eval_summary.json"
# USER_GUIDANCE_PATH = "./configs/guidance_causalbank.json"
#
# !python ../../experiments/methods/run_neoolaf.py \
#   --dataset-jsonl-path "{DATASET_JSONL}" \
#   --ontology-path "{ONTOLOGY_PATH}" \
#   --output-jsonl-path "{OUTPUT_JSONL}" \
#   --backend-name "{BACKEND}" \
#   --host "{HOST}" \
#   --api-key "{API_KEY}" \
#   --model-name "{MODEL_NAME}" \
#   --max-tokens {MAX_TOKENS} \
#   --openrouter-reasoning-effort {OPENROUTER_REASONING_EFFORT} \
#   --openrouter-exclude-reasoning \
#   --type-filter all \
#   --user-guidance-path "{USER_GUIDANCE_PATH}" \
#   --few-shot-from-dataset \
#   --few-shot-source-type all \
#   --few-shot-k 1 \
#   --output-format canonical \
#   --artifacts-root "./runs/neoolaf_causalbank_artifacts" \
#   --chunk-size {CHUNK_SIZE} \
#   --chunk-overlap {CHUNK_OVERLAP} \
#   --max-chunks {MAX_CHUNKS} \
#   --max-expressions {MAX_EXPRESSIONS} \
#   --max-relation-mentions {MAX_RELATION_MENTIONS} \
#   --max-workers {MAX_WORKERS} \
#   --document-workers {DOCUMENT_WORKERS} \
#   --no-web-search \
#   --no-checkpoints \
#   --no-chunk-checkpoints \
#   --no-resume \
#   {MAX_DOCS_ARG}
#
# !python ../../experiments/evaluation/eval_relations.py \
#   --gold "{EVAL_GOLD_JSONL}" \
#   --pred "{OUTPUT_JSONL}" \
#   --output "{SUMMARY_OUTPUT}"

## EventStoryLine, commented

In [ ]:
# DATASET_JSONL = "../../../ragtree/data/preprocessed/eventstoryline.jsonl"
# ONTOLOGY_PATH = "../../../ragtree/data/ontology/OWLTime/time.ttl"
# OUTPUT_JSONL = "./runs/neoolaf_eventstoryline_predictions.canonical.jsonl"
# SUMMARY_OUTPUT = "./runs/neoolaf_eventstoryline_eval_summary.json"
# USER_GUIDANCE_PATH = "./configs/guidance_eventstoryline.json"
#
# !python ../../experiments/methods/run_neoolaf.py \
#   --dataset-jsonl-path "{DATASET_JSONL}" \
#   --ontology-path "{ONTOLOGY_PATH}" \
#   --output-jsonl-path "{OUTPUT_JSONL}" \
#   --backend-name "{BACKEND}" \
#   --host "{HOST}" \
#   --api-key "{API_KEY}" \
#   --model-name "{MODEL_NAME}" \
#   --max-tokens {MAX_TOKENS} \
#   --openrouter-reasoning-effort {OPENROUTER_REASONING_EFFORT} \
#   --openrouter-exclude-reasoning \
#   --type-filter all \
#   --user-guidance-path "{USER_GUIDANCE_PATH}" \
#   --few-shot-from-dataset \
#   --few-shot-source-type all \
#   --few-shot-k 1 \
#   --output-format canonical \
#   --artifacts-root "./runs/neoolaf_eventstoryline_artifacts" \
#   --chunk-size {CHUNK_SIZE} \
#   --chunk-overlap {CHUNK_OVERLAP} \
#   --max-chunks {MAX_CHUNKS} \
#   --max-expressions {MAX_EXPRESSIONS} \
#   --max-relation-mentions {MAX_RELATION_MENTIONS} \
#   --max-workers {MAX_WORKERS} \
#   --document-workers {DOCUMENT_WORKERS} \
#   --no-web-search \
#   --no-checkpoints \
#   --no-chunk-checkpoints \
#   --no-resume \
#   {MAX_DOCS_ARG}
#
# !python ../../experiments/evaluation/eval_relations.py \
#   --gold "{EVAL_GOLD_JSONL}" \
#   --pred "{OUTPUT_JSONL}" \
#   --output "{SUMMARY_OUTPUT}"

## FinCausal, commented

In [ ]:
# DATASET_JSONL = "../../../ragtree/data/preprocessed/fincausal.jsonl"
# ONTOLOGY_PATH = "../../../ragtree/data/ontology/FIBO-CorePlus/fibo-core-plus.ttl"
# OUTPUT_JSONL = "./runs/neoolaf_fincausal_predictions.canonical.jsonl"
# SUMMARY_OUTPUT = "./runs/neoolaf_fincausal_eval_summary.json"
# USER_GUIDANCE_PATH = "./configs/guidance_fincausal.json"
#
# !python ../../experiments/methods/run_neoolaf.py \
#   --dataset-jsonl-path "{DATASET_JSONL}" \
#   --ontology-path "{ONTOLOGY_PATH}" \
#   --output-jsonl-path "{OUTPUT_JSONL}" \
#   --backend-name "{BACKEND}" \
#   --host "{HOST}" \
#   --api-key "{API_KEY}" \
#   --model-name "{MODEL_NAME}" \
#   --max-tokens {MAX_TOKENS} \
#   --openrouter-reasoning-effort {OPENROUTER_REASONING_EFFORT} \
#   --openrouter-exclude-reasoning \
#   --type-filter all \
#   --user-guidance-path "{USER_GUIDANCE_PATH}" \
#   --few-shot-from-dataset \
#   --few-shot-source-type all \
#   --few-shot-k 1 \
#   --output-format canonical \
#   --artifacts-root "./runs/neoolaf_fincausal_artifacts" \
#   --chunk-size {CHUNK_SIZE} \
#   --chunk-overlap {CHUNK_OVERLAP} \
#   --max-chunks {MAX_CHUNKS} \
#   --max-expressions {MAX_EXPRESSIONS} \
#   --max-relation-mentions {MAX_RELATION_MENTIONS} \
#   --max-workers {MAX_WORKERS} \
#   --document-workers {DOCUMENT_WORKERS} \
#   --no-web-search \
#   --no-checkpoints \
#   --no-chunk-checkpoints \
#   --no-resume \
#   {MAX_DOCS_ARG}
#
# !python ../../experiments/evaluation/eval_relations.py \
#   --gold "{EVAL_GOLD_JSONL}" \
#   --pred "{OUTPUT_JSONL}" \
#   --output "{SUMMARY_OUTPUT}"

## MAVEN-ERE, commented

In [ ]:
# DATASET_JSONL = "../../../ragtree/data/preprocessed/maven_ere.jsonl"
# ONTOLOGY_PATH = "../../../ragtree/data/ontology/EventKG/EventKGSchema.ttl"
# OUTPUT_JSONL = "./runs/neoolaf_maven_ere_predictions.canonical.jsonl"
# SUMMARY_OUTPUT = "./runs/neoolaf_maven_ere_eval_summary.json"
# USER_GUIDANCE_PATH = "./configs/guidance_maven_ere.json"
#
# !python ../../experiments/methods/run_neoolaf.py \
#   --dataset-jsonl-path "{DATASET_JSONL}" \
#   --ontology-path "{ONTOLOGY_PATH}" \
#   --output-jsonl-path "{OUTPUT_JSONL}" \
#   --backend-name "{BACKEND}" \
#   --host "{HOST}" \
#   --api-key "{API_KEY}" \
#   --model-name "{MODEL_NAME}" \
#   --max-tokens {MAX_TOKENS} \
#   --openrouter-reasoning-effort {OPENROUTER_REASONING_EFFORT} \
#   --openrouter-exclude-reasoning \
#   --type-filter all \
#   --user-guidance-path "{USER_GUIDANCE_PATH}" \
#   --few-shot-from-dataset \
#   --few-shot-source-type all \
#   --few-shot-k 1 \
#   --output-format canonical \
#   --artifacts-root "./runs/neoolaf_maven_ere_artifacts" \
#   --chunk-size {CHUNK_SIZE} \
#   --chunk-overlap {CHUNK_OVERLAP} \
#   --max-chunks {MAX_CHUNKS} \
#   --max-expressions {MAX_EXPRESSIONS} \
#   --max-relation-mentions {MAX_RELATION_MENTIONS} \
#   --max-workers {MAX_WORKERS} \
#   --document-workers {DOCUMENT_WORKERS} \
#   --no-web-search \
#   --no-checkpoints \
#   --no-chunk-checkpoints \
#   --no-resume \
#   {MAX_DOCS_ARG}
#
# !python ../../experiments/evaluation/eval_relations.py \
#   --gold "{EVAL_GOLD_JSONL}" \
#   --pred "{OUTPUT_JSONL}" \
#   --output "{SUMMARY_OUTPUT}"